# End-to-End RAG Evaluation

Measure retrieval (hit rate, MRR) and generation (correctness, faithfulness) over a golden set.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

**Extra requirements:** pandas

Uses the shared helpers in `../common.py` (local embeddings + Ollama). Make sure `ollama serve` is running and `ollama pull llama3.2` is done.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

In [ ]:
import pandas as pd
questions = load_eval_questions()
print(len(questions), "golden questions")

## Retrieval metrics
- **Hit rate@k** - was the evidence retrieved at all?
- **MRR** - how high did it rank?

In [ ]:
def retrieval_metrics(k=5):
    hits_k, rr = [], []
    for q in questions:
        results = store.search(q["question"], k)
        rank = next((i + 1 for i, h in enumerate(results) if q["evidence"].lower() in h["text"].lower()), None)
        hits_k.append(rank is not None)
        rr.append(1 / rank if rank else 0)
    return {"hit_rate@5": sum(hits_k) / len(hits_k), "MRR": sum(rr) / len(rr)}

retrieval_metrics()

## Generation metrics
LLM-as-judge for **correctness** (vs the reference answer) and **faithfulness** (vs the retrieved context).

In [ ]:
rows = []
for q in questions:
    response, hits = rag(q["question"], store, k=3)
    ctx = "\n".join(h["text"] for h in hits)
    rows.append({
        "question": q["question"][:55],
        "correct": llm_yes(f"Reference answer: {q['answer']}\nCandidate answer: {response}\nDoes the candidate agree with the reference answer?"),
        "faithful": llm_yes(f"Context:\n{ctx}\n\nAnswer: {response}\nIs every claim in the answer supported by the context?"),
    })
report = pd.DataFrame(rows)
display(report)
print(report[["correct", "faithful"]].mean())

## Synthetic questions
No golden set yet? Have the LLM write question-answer pairs from your own chunks, then review them by hand.

In [ ]:
def synth_qa(chunk):
    return llm_json('Write one factual question answered by the passage. Reply as JSON {"question": str, "answer": str}.\n\nPassage: ' + chunk)

print(synth_qa(chunks[5]))